# 12 Hidden factors and reliability: Holzinger and Swineford's nine ability tests

The scores of 301 pupils of two Chicago schools on nine mental ability tests (Holzinger & Swineford 1939), in
the nine-test version that ships with R lavaan (`HolzingerSwineford1939`), checked with

- `table.check_hidden_factors` (confirmatory factor analysis, R lavaan): the three-factor model (visual: x1-x3,
  textual: x4-x6, speed: x7-x9) whose fit Rosseel (2012, Journal of Statistical Software 48(2), 1-36) prints:
  chi-square 85.306 on 24 degrees of freedom, CFI 0.931, TLI 0.896, RMSEA 0.092 (90 % CI 0.071 to 0.114),
  SRMR 0.065; a structural path (`formula=`); and full-information maximum likelihood with empty cells;
- `table.find_hidden_factors_behind` (exploratory factor analysis, R psych with GPArotation): parallel analysis,
  minres with oblimin, principal axis with promax, maximum likelihood with varimax;
- `table.reliability_of` (Cronbach's alpha with Feldt's interval, R psych) for the three groups of three tests.

**Only simplydata runs in this notebook.** The reference values (`references/12_factor_analysis.csv`, from
`scripts/references/12_factor_analysis.py`), each check printing its source: the printed values (half a unit of the last printed digit); R lavaan, psych and base R `factanal`,
run here; statsmodels `Factor(method="ml")` (Python); and the closed formulas for Cronbach's alpha (Cronbach
1951) and Feldt's (1965) interval, computed with NumPy and SciPy.

Tolerances (the same for every dataset): 1e-6 relative for results of an iterative fit against the same program
run the same way (lavaan, psych; as README); 1e-12 for alpha against its formula and 1e-10 for Feldt's interval
(a closed formula with F quantiles); 1e-3 for maximum likelihood fitted by a *different* optimiser (base R
`factanal`, statsmodels), which stops at a slightly different point of the same likelihood (set before running,
from the size of the stopping rules; it is not a loosened tolerance). psych's parallel analysis simulates data and
GPArotation's oblimin starts from random numbers; simplydata calls `set.seed(with_seed)` (1) first, and the R
reference does the same in the same order. The loadings are compared as psych gives them (its factor order and
signs), which simplydata keeps, so no alignment is needed.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_rows, versions

NOTEBOOK = "12_factor_analysis"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def rows_by(column, word, *args, **options):
    return {row[column]: row for row in read_rows(quiet(word, *args, **options))}


hs = load("holzinger_swineford")
TESTS = [f"x{i}" for i in range(1, 10)]
MEASURED_BY = {"visual": ["x1", "x2", "x3"], "textual": ["x4", "x5", "x6"], "speed": ["x7", "x8", "x9"]}
DATASET = "Holzinger & Swineford 1939"

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit 3f0192b of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## The three-factor model of the lavaan paper: `check_hidden_factors`

The full printed answer:

In [2]:
answer = hs.check_hidden_factors(measured_by=MEASURED_BY)

Confirmatory factor analysis of 3 hidden factors measured by 9 columns (301 rows): visual by x1, x2 and x3; textual by x4, x5 and x6; speed by x7, x8 and x9.
If the model were exactly right, a gap between the model and the data this big or bigger would turn up in fewer than 1 in 1000 studies like this one (chi-square 85.3 with 24 degrees of freedom, p < 0.0001).
A large p-value does not show the model is right; a small one is expected with many rows (300 or more) even for a useful model. Read the guides below.
  CFI 0.931: how much better the model matches the data than a model in which the columns are unrelated (1 = as well as possible).
  TLI 0.896: the same idea, with a penalty for each extra value the model estimates.
  RMSEA 0.092 (90 % CI 0.071 to 0.114): the misfit for each degree of freedom (0 = none).
  SRMR 0.065: the average gap between the correlations the model implies and those in the data.
  Commonly used guides: CFI and TLI 0.95 or more, RMSEA 0.06 or less, SRMR 0.08 or

In [3]:
found = {row["term"]: row for row in read_rows(answer)}
values = {"chi-square": found["chi-square"]["estimate"],
          "degrees of freedom": found["chi-square"]["degrees of freedom"],
          "chi-square p-value": found["chi-square"]["p-value"],
          "CFI": found["CFI"]["estimate"], "TLI": found["TLI"]["estimate"],
          "RMSEA": found["RMSEA"]["estimate"], "RMSEA 90 % CI low": found["RMSEA"]["90 % CI low"],
          "RMSEA 90 % CI high": found["RMSEA"]["90 % CI high"], "SRMR": found["SRMR"]["estimate"]}
for term, row in found.items():
    if " measured by " in term or " with " in term:
        values[f"{term} (standardised)"] = row["estimate"]
        values[f"{term} standard error"] = row["standard error"]
checks.check_values(DATASET, "check_hidden_factors (measured_by=)", values)

agrees  Holzinger & Swineford 1939 | check_hidden_factors (measured_by=) | chi-square: simplydata 85.30552177, Rosseel (2012), printed 85.306; difference 4.78e-04 (allowed 5e-04)
      Source: Rosseel (2012), printed (printed) - https://doi.org/10.18637/jss.v048.i02
agrees  Holzinger & Swineford 1939 | check_hidden_factors (measured_by=) | chi-square: simplydata 85.30552177, R lavaan::fitMeasures 85.30552177; relative difference 0.00e+00 (allowed 1e-06)
      Source: R lavaan::fitMeasures (run reference) - https://search.r-project.org/CRAN/refmans/lavaan/html/fitMeasures.html (version 0.7.3)
agrees  Holzinger & Swineford 1939 | check_hidden_factors (measured_by=) | degrees of freedom: simplydata 24, Rosseel (2012), printed 24; difference 0.00e+00 (allowed 0e+00)
      Source: Rosseel (2012), printed (printed) - https://doi.org/10.18637/jss.v048.i02
agrees  Holzinger & Swineford 1939 | check_hidden_factors (measured_by=) | degrees of freedom: simplydata 24, R lavaan::fitMeasures 24; rel

41

## A structural path, and empty cells

- `formula="visual =~ x1 + x2 + x3; speed =~ x7 + x8 + x9; speed ~ visual"`: R `lavaan::sem`; the standardised
  path, its standard error, the chi-square and CFI.
- The first 30 scores of x1 emptied on purpose: simplydata keeps every row (full-information maximum likelihood,
  as lavaan's `missing = "ml"`).

In [4]:
found = rows_by("term", hs.check_hidden_factors, formula="visual =~ x1 + x2 + x3; speed =~ x7 + x8 + x9; speed ~ visual")
checks.check_values(DATASET, "check_hidden_factors (formula=)",
                    {"speed explained by visual": found["speed explained by visual"]["estimate"],
                     "its standard error": found["speed explained by visual"]["standard error"],
                     "chi-square": found["chi-square"]["estimate"], "CFI": found["CFI"]["estimate"],
                     "RMSEA": found["RMSEA"]["estimate"]})

emptied = load("holzinger_swineford_x1_emptied")
found = rows_by("term", emptied.check_hidden_factors, measured_by=MEASURED_BY)
checks.check_values(DATASET + ", 30 cells of x1 emptied", "check_hidden_factors (empty cells, FIML)",
                    {"chi-square": found["chi-square"]["estimate"], "CFI": found["CFI"]["estimate"],
                     "RMSEA": found["RMSEA"]["estimate"],
                     "visual measured by x1 (standardised)": found["visual measured by x1"]["estimate"],
                     "speed measured by x9 (standardised)": found["speed measured by x9"]["estimate"]})

agrees  Holzinger & Swineford 1939 | check_hidden_factors (formula=) | speed explained by visual: simplydata 0.460317218924, R lavaan::sem 0.460317218924; relative difference 0.00e+00 (allowed 1e-06)
      Source: R lavaan::sem (run reference) - https://search.r-project.org/CRAN/refmans/lavaan/html/sem.html (version 0.7.3)
agrees  Holzinger & Swineford 1939 | check_hidden_factors (formula=) | its standard error: simplydata 0.0741697800268, R lavaan::sem 0.0741697800268; relative difference 1.87e-16 (allowed 1e-06)
      Source: R lavaan::sem (run reference) - https://search.r-project.org/CRAN/refmans/lavaan/html/sem.html (version 0.7.3)
agrees  Holzinger & Swineford 1939 | check_hidden_factors (formula=) | chi-square: simplydata 47.4133581633, R lavaan::sem 47.4133581633; relative difference 0.00e+00 (allowed 1e-06)
      Source: R lavaan::sem (run reference) - https://search.r-project.org/CRAN/refmans/lavaan/html/sem.html (version 0.7.3)
agrees  Holzinger & Swineford 1939 | check_hidd

agrees  Holzinger & Swineford 1939, 30 cells of x1 emptied | check_hidden_factors (empty cells, FIML) | chi-square: simplydata 87.4018469694, R lavaan::cfa(missing = 'ml') 87.4018469694; relative difference 0.00e+00 (allowed 1e-06)
      Source: R lavaan::cfa(missing = 'ml') (run reference) - https://search.r-project.org/CRAN/refmans/lavaan/html/cfa.html (version 0.7.3)
agrees  Holzinger & Swineford 1939, 30 cells of x1 emptied | check_hidden_factors (empty cells, FIML) | CFI: simplydata 0.927941793738, R lavaan::cfa(missing = 'ml') 0.927941793738; relative difference 0.00e+00 (allowed 1e-06)
      Source: R lavaan::cfa(missing = 'ml') (run reference) - https://search.r-project.org/CRAN/refmans/lavaan/html/cfa.html (version 0.7.3)
agrees  Holzinger & Swineford 1939, 30 cells of x1 emptied | check_hidden_factors (empty cells, FIML) | RMSEA: simplydata 0.0936832796685, R lavaan::cfa(missing = 'ml') 0.0936832796685; relative difference 7.41e-16 (allowed 1e-06)
      Source: R lavaan::cfa(

5

## Exploratory factor analysis: `find_hidden_factors_behind`

The defaults: parallel analysis (Horn 1965) chooses how many factors, then minres with oblimin. Every loading
(9 tests x 3 factors), shared part (communality) and own part (uniqueness) is compared with R `psych::fa` run the
same way after `set.seed(1)`. Then `how_many=3` with principal axis factoring and promax, and maximum likelihood
with varimax; the own parts of maximum likelihood do not depend on the rotation and are also compared with base
R `factanal` and statsmodels `Factor(method="ml")`, two other optimisers of the same likelihood.

In [5]:
hs.find_hidden_factors_behind(TESTS)

Factor analysis of 9 columns (301 rows): 3 hidden factors, minimum residual (minres), oblimin rotation (the factors may correlate).
Parallel analysis suggests 3 factors: 3 stand out more than they would in random data of the same size (Horn 1965; with_seed=1).
Factor analysis looks for a few hidden factors that the columns share; principal components (add_principal_components) only sum the columns up.
Loadings (how strongly each column goes with each factor, usually -1 to 1; below 0.3 left blank here, all of them in the table given back), and each column's shared part (communality):
      factor 1  factor 2  factor 3  shared part
  x1                0.59                   0.48
  x2                0.51                   0.26
  x3                0.69                   0.45
  x4      0.85                             0.73
  x5      0.89                             0.75
  x6      0.81                             0.69
  x7                          0.74         0.52
  x8                      

,column,factor 1,factor 2,factor 3,shared part,own part
1,x1,0.196439,0.591615,0.030951,0.476754,0.523246
2,x2,0.042971,0.508926,-0.122300,0.255227,0.744773
3,x3,-0.062294,0.685927,0.019043,0.453450,0.546550
4,x4,0.846103,0.016270,0.007664,0.727936,0.272064
5,x5,0.885518,-0.064592,0.006715,0.753731,0.246269
6,x6,0.805257,0.079619,-0.012975,0.691355,0.308645
7,x7,0.043791,-0.152327,0.736544,0.518555,0.481445
8,x8,-0.034007,0.125105,0.686497,0.520172,0.479828
9,x9,0.031835,0.382276,0.456360,0.460461,0.539539


In [6]:
def efa(label, **options):
    found = rows_by("column", hs.find_hidden_factors_behind, TESTS, **options)
    factors = [c for c in found[TESTS[0]] if c.startswith("factor ")]
    values = {"number of factors": len(factors)}
    for test in TESTS:
        for factor in factors:
            values[f"{test} on {factor}"] = found[test][factor]
        values[f"shared part of {test}"] = found[test]["shared part"]
        values[f"own part of {test}"] = found[test]["own part"]
    checks.check_values(DATASET, f"find_hidden_factors_behind ({label})", values)


efa("defaults: parallel analysis, minres, oblimin")
efa("3 factors, principal axis, promax", how_many=3, method="principal axis", rotation="promax")
efa("3 factors, maximum likelihood, varimax", how_many=3, method="ml", rotation="varimax")

agrees  Holzinger & Swineford 1939 | find_hidden_factors_behind (defaults: parallel analysis, minres, oblimin) | number of factors: simplydata 3, R psych 3; difference 0.00e+00 (allowed 0e+00)
      Source: R psych (run reference) - https://cran.r-project.org/package=psych (version 2.6.9)
agrees  Holzinger & Swineford 1939 | find_hidden_factors_behind (defaults: parallel analysis, minres, oblimin) | x1 on factor 1: simplydata 0.196439133949, R psych::fa 0.196439133949; relative difference 2.83e-16 (allowed 1e-06)
      Source: R psych::fa (run reference) - https://search.r-project.org/CRAN/refmans/psych/html/fa.html (version 2.6.9)
agrees  Holzinger & Swineford 1939 | find_hidden_factors_behind (defaults: parallel analysis, minres, oblimin) | x1 on factor 2: simplydata 0.591615227617, R psych::fa 0.591615227617; relative difference 1.88e-16 (allowed 1e-06)
      Source: R psych::fa (run reference) - https://search.r-project.org/CRAN/refmans/psych/html/fa.html (version 2.6.9)
agrees  Ho

## Cronbach's alpha with Feldt's interval: `reliability_of`

For each group of three tests: alpha = k / (k - 1) (1 - sum of the item variances / variance of the sum)
(Cronbach 1951), alpha without each item from the same formula on the other two, and Feldt's (1965) 95 % interval
1 - (1 - alpha) F, with F the 97.5 % and 2.5 % quantiles of the F distribution on n - 1 and (n - 1)(k - 1)
degrees of freedom (SciPy). R: `psych::alpha` and `psych::alpha.ci`.

In [7]:
hs.reliability_of(["x4", "x5", "x6"])

Cronbach's alpha = 0.883 (95 % CI 0.858 to 0.904, Feldt) for 3 items and 301 rows.
No single item raises alpha when it is left out (the highest without one item is 0.843, without "x6").
Alpha rises with the number of items and does not show that the items measure one thing. Method: R psych::alpha() (raw alpha) and psych::alpha.ci() (Feldt's interval).


,measure,estimate,estimate 95 % CI low,estimate 95 % CI high,items,rows,method
1,Cronbach's alpha,0.882707,0.857731,0.903925,3,301,"R psych::alpha(), psych::alpha.ci() (Feldt)"
2,alpha without x4,0.830693,NaN,NaN,2,301,R psych::alpha() alpha.drop
3,alpha without x5,0.825730,NaN,NaN,2,301,R psych::alpha() alpha.drop
4,alpha without x6,0.843459,NaN,NaN,2,301,R psych::alpha() alpha.drop


In [8]:
for factor, items in MEASURED_BY.items():
    found = rows_by("measure", hs.reliability_of, items)
    values = {"Cronbach's alpha": found["Cronbach's alpha"]["estimate"],
              "95 % CI low": found["Cronbach's alpha"]["estimate 95 % CI low"],
              "95 % CI high": found["Cronbach's alpha"]["estimate 95 % CI high"]}
    for item in items:
        values[f"alpha without {item}"] = found[f"alpha without {item}"]["estimate"]
    checks.check_values(f"{DATASET}, {factor} tests", "reliability_of", values)

agrees  Holzinger & Swineford 1939, visual tests | reliability_of | Cronbach's alpha: simplydata 0.626117131885, Cronbach (1951) formula 0.626117131885; relative difference 0.00e+00 (allowed 1e-12)
      Source: Cronbach (1951) formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/12_factor_analysis.py
agrees  Holzinger & Swineford 1939, visual tests | reliability_of | Cronbach's alpha: simplydata 0.626117131885, R psych::alpha 0.626117131885; relative difference 0.00e+00 (allowed 1e-12)
      Source: R psych::alpha (run reference) - https://search.r-project.org/CRAN/refmans/psych/html/alpha.html (version 2.6.9)
agrees  Holzinger & Swineford 1939, visual tests | reliability_of | 95 % CI low: simplydata 0.546503337103, Feldt (1965) formula (SciPy F) 0.546503337103; relative difference 2.03e-16 (allowed 1e-10)
      Source: Feldt (1965) formula (SciPy F) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.

## Result

In [9]:
results = checks.finish()

12_factor_analysis: 243 comparisons - 243 agrees
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)
